# Spectrogram Conversion & Spectral Gating

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torchaudio
import torchaudio.transforms as T
import torchaudio.functional as F

## Spectrogram Parameters

- **TARGET**: Sample rate in Hz (16 kHz = 16,000 samples/sec)
- **N_FFT**: FFT window size in samples (400 samples = 25 ms at 16 kHz)
- **HOP_LENGTH**: Step size between windows in samples (160 samples = 10 ms)
- **N_MELS**: Number of Mel-frequency filter banks

In [ ]:
TARGET = 16000
N_FTT = 400       # 25ms window at 16kHz
HOP_LENGTH = 160  # 10ms
N_MELS = 40

## Spectrogram Conversion


### Compute Short-Time-Fourier Transform
This function would convert a raw waveform into a complex-valued STFT spectrogram using a Hann window. In the end, it should have a shape of `(n_freqs, n_frames)` where `n_freqs = N_FFT // 2 + 1 = 201` frequency bins.

In [ ]:
def compute_stft(waveform: torch.Tensor) -> torch.Tensor:
    """Compute complex STFT spectrogram.

    Args:
        waveform (torch.Tensor): 

    Returns:
        torch.Tensor: 
    """
    if waveform.ndim == 2:
        waveform = waveform.squeeze(0) # (n_channels, n_samples) -> (n_samples,)
    
    window = torch.hann_window(N_FTT, device=waveform.device) # (n_fft,)
    
    stft = torch.stft(
        waveform,
        n_fft=N_FTT,
        hop_length=HOP_LENGTH,
        win_length=N_FTT,
        window=window,
        return_complex=True
    )
    return stft  # (n_freqs, n_frames)

In [ ]:
# Test
waveform = torch.randn(TARGET)
stft = compute_stft(waveform)
print(f"Input shape: {waveform.shape}")
print(f"STFT shape: {stft.shape}")
print(f"Expected: ({N_FTT // 2 + 1}, {TARGET // HOP_LENGTH + 1})")
print(f"STFT dtype: {stft.dtype}")

### Power Spectrogram

Computes the power spectrogram by taking the squared magnitude of the complex STFT. During the discards phase, this would represent the enery at each frequency bin per frame. 

In [ ]:
def compute_power_spectrogram(waveform: torch.Tensor) -> torch.Tensor:
    """Compute power spectrogram.

    Args:
        waveform (torch.Tensor): 

    Returns:
        torch.Tensor: 
    """
    stft = compute_stft(waveform)
    power_spec = torch.abs(stft) ** 2
    return power_spec  # (n_freqs, n_frames)    

In [ ]:
# Test
power_spec = compute_power_spectrogram(waveform)
print(f"Power spectrogram shape: {power_spec.shape}")
print(f"Min: {power_spec.min().item():.4e}, Max: {power_spec.max().item():.4e}")
print(f"Mean: {power_spec.mean().item():.4e}, Std: {power_spec.std().item():.4e}")
print(f"Expected: ({N_FTT // 2 + 1}, {TARGET // HOP_LENGTH + 1})")

In [ ]:
# Visualize
plt.figure(figsize=(10, 4))
plt.imshow(10 * torch.log10(power_spec + 1e-10).numpy(), aspect='auto', origin='lower')
plt.colorbar(label="Log Power (dB)")
plt.xlabel("Time Frames")
plt.ylabel("Frequency Bin (Hz)")
plt.title("Power Spectrogram")
plt.tight_layout()
plt.show()

## Frequency-Domain Noise Suppression (Spectral Gating)

### Estimate the noise profile
This would estimates the noise profile by averaging the power spectrogram over the first `n_frames` frames. This is under the assumption the beginning of the clip contains mostly background noise. This noise estimate will be used later in the spectral subtraction to suprress stationary noise while preserving the speech content.

In [ ]:
def estimate_noise(
    power_spec: torch.Tensor, 
    n_frames: int = 5
    ) -> torch.Tensor:
    """Estimate noise power spectrogram by averaging the first n_frames.

    Args:
        power_spec (torch.Tensor): Power spectrogram of shape (n_freqs, n_frames_total).
        n_frames (int, optional): Number of initial frames to average. Defaults to 5.

    Returns:
        torch.Tensor: Estimated noise power spectrogram of shape (n_freqs,).
    """
    return power_spec[:, :n_frames].mean(dim=1)  # (n_freqs,)
    

In [ ]:
# Test
noise_profile = estimate_noise(power_spec)
print(f"Noise profile shape: {noise_profile.shape}")
print(f"Min: {noise_profile.min().item():.4e}, Max: {noise_profile.max().item():.4e}")

In [ ]:
# Visualize noise profile
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Show which frames are used
ax = axes[0]
ax.imshow(10 * torch.log10(power_spec + 1e-10).numpy(), aspect='auto', origin='lower')
ax.axvline(x=5, color='red', linestyle='--', label='Noise Frames')
ax.axvspan(0, 5, color='red', alpha=0.3)
ax.set_xlabel("Time Frames")
ax.set_ylabel("Frequency Bin (Hz)")
ax.set_title("Frames Used for Noise Estimation")
ax.legend()

# Show the profile
ax = axes[1]
freq_bins = np.arange(noise_profile.shape[0])
ax.barh(freq_bins, noise_profile.numpy(), color='blue')
ax.set_xlabel("Estimated Noise Power")
ax.set_ylabel("Frequency Bin (Hz)")
ax.set_title("Estimated Noise Profile |\hat{N}(f)|^2")

plt.tight_layout()
plt.show()

### Spectral Subtraction
This would subtracts the estimated noise profile from the power spectrogram, then scaled by an over-subtraction factor of `omega` to handle noise underestimation. Afterward, a spectral floor of `delta * power_spec` prevents negative energy values and reduces the musical noise artificats. In the end, the result will show a clean power spectrogram ready for the Mel converison. 

In [ ]:
def spectral_subtraction(
    power_spec: torch.Tensor, 
    noise_profile: torch.Tensor,
    omega: float = 1.5,
    delta: float = 0.5
    ) -> torch.Tensor:
    """Perform spectral subtraction.

    Args:
        power_spec (torch.Tensor): Power spectrogram of shape (n_freqs, n_frames).
        noise_profile (torch.Tensor): Noise power spectrogram of shape (n_freqs,).
        omega (float, optional): Scaling factor for noise profile. Defaults to 1.5.
        delta (float, optional): Minimum value for clamping. Defaults to 0.5.

    Returns:
        torch.Tensor: Denoised power spectrogram of shape (n_freqs, n_frames).
    """
    subtracted = power_spec - omega * noise_profile
    floor = delta * power_spec
    return torch.max(subtracted, floor)  # (n_freqs, n_frames)

In [ ]:
# Test on noisy sample
# Insert musan noise injection

noisy_power = compute_power_spectrogram(noisy)
noisy_noise_profile = estimate_noise(noisy_power, noise_frame = 5)
cleaned_power = spectral_subtraction(noisy_power, noisy_noise_profile)

print(f"Noisy power spectrogram shape: {noisy_power.shape}")
print(f"Cleaned power spectrogram shape: {cleaned_power.shape}")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.imshow(torch.log10(noisy_power + 1e-10).numpy(), aspect='auto', origin='lower')
ax.set_title("Noisy Power Spectrogram")
ax.set_xlabel("Time Frames")
ax.set_ylabel("Frequency Bin (Hz)")

ax = axes[1]
ax.imshow(torch.log10(cleaned_power + 1e-10).numpy(), aspect='auto', origin='lower')
ax.set_title("Cleaned Power Spectrogram")
ax.set_xlabel("Time Frames")
ax.set_ylabel("Frequency Bin (Hz)")

ax = axes[2]
diff = noisy_power - cleaned_power
ax.imshow(torch.log10(diff + 1e-10).numpy(), aspect='auto', origin='lower')
ax.set_title("Difference (Noise Removed)")
ax.set_xlabel("Time Frames")

plt.suptitle("Spectral Subtraction at SNR = 5 dB")
plt.tight_layout()
plt.show()

### Apply Spectral Gating

This would be the end-to-end wrapper that chains the full spectral gating pipeline. It would computes the power spectrogram, then estimate the noise profile, and apply the spectral subtraction. Afterward, it would reconstructs the denoised waveform using the inverse STFT. This would produces a cleaned time-domain signal that can be passed into the Mel spectrogram stage.

In [ ]:
def apply_spectral_gating(
    waveform: torch.Tensor,
    omega: float = 1.5,
    delta: float = 0.02,
    noise_frame: int = 5
    ) -> torch.Tensor:
    """Apply spectral gating.

    Args:
        waveform (torch.Tensor): 
        omega (float, optional): Defaults to 1.5.
        delta (float, optional): Defaults to 0.02.
        noise_frame (int, optional): Defaults to 5.

    Returns:
        torch.Tensor: 
    """
    power_spec = compute_power_spectrogram(waveform)
    noise_profile = estimate_noise(power_spec, noise_frame)
    denoised = spectral_subtraction(power_spec, noise_profile, omega, delta)
    return torch.istft(denoised, hop_length=HOP_LENGTH, win_length=N_FTT)

In [ ]:
# Test
cleaned = apply_spectral_gating(noisy)
print(f"Cleaned waveform shape: {cleaned.shape}")

In [ ]:
# Compare clean vs noisy vs gated
cleaned_pwoer = compute_power_spectrogram(waveform)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ["Clean Power Spectrogram", "Noisy Power Spectrogram", "Gated Power Spectrogram"]
powers = [cleaned_pwoer, noisy_power, cleaned_power]

for ax, title, power in zip(axes, titles, powers):
    ax.imshow(torch.log10(power + 1e-10).numpy(), aspect='auto', origin='lower')
    ax.set_title(title)
    ax.set_xlabel("Time Frames")
    ax.set_ylabel("Frequency Bin (Hz)")
    
plt.suptitle("Spectral Gating Comparison")
plt.tight_layout()
plt.show()

### Bandpass Filter (Ablation Baseline)

This would apply a fixed bandpass filter which would retain only 300-3400 Hz speech band. This serves as a simpler ablation baseline compared to the spectral gating, and would remove out-of-band noise, but cannot be adapt to the noise profile within the speech fequencies. 

In [ ]:
def apply_bandpass(
    waveform: torch.Tensor, 
    low_freq: float = 300.0, 
    high_freq: float = 3400.0, 
    sample_rate: int = 16000
    ) -> torch.Tensor:
    """Apply bandpass filter to the waveform.

    Args:
        waveform (torch.Tensor): Input audio waveform of shape (n_samples,).
        low_freq (float, optional): Low cutoff frequency in Hz. Defaults to 300.0.
        high_freq (float, optional): High cutoff frequency in Hz. Defaults to 3400.0.
        sample_rate (int, optional): Sample rate of the audio in Hz. Defaults to 16000.

    Returns:
        torch.Tensor: Bandpass filtered waveform of shape (n_samples,).
    """
    filtered = F.highpass_biquad(waveform, sample_rate, low_freq)
    filtered = F.lowpass_biquad(filtered, sample_rate, high_freq)
    return filtered

In [ ]:
# Test
bandpass_wav = apply_bandpass(noisy)
print(f"Input shape: {noisy.shape}, Bandpass output shape: {bandpass_wav.shape}")

# Audio Comparison
print("\nAudio Comparison:")
display(Audio(noisy.numpy(), rate=TARGET))
print("Bandpass Filtered Audio:")
display(Audio(bandpass_wav.numpy(), rate=TARGET))

### Log-Mel Spectrogram

This would maps the power spectrogram onto a 40-band Mel-Frequency scale, then would converts to decibels. The Mel scale approximates human auditory preception, and would compress higher frequencies where the ear is less sensitive . The log-Mel spectrogram is the final feature representation fed into the CNN and AST classifiers.

In [ ]:
def power_to_log_mel(power_spec: torch.Tensor) -> torch.Tensor:
    """Convert power spectrogram to log-mel spectrogram.

    Args:
        power_spec (torch.Tensor): Power spectrogram of shape (n_freqs, n_frames).

    Returns:
        torch.Tensor: Log-mel spectrogram of shape (n_mels, n_frames).
    """
    mel_scale = T.MelScale(n_mels=N_MELS, sample_rate=TARGET, n_stft=N_FTT // 2 + 1)
    
    if power_spec.ndim == 2:
        power_spec = power_spec.unsqueeze(0)      # (1, n_freqs, n_frames)
        
    mel = mel_scale(power_spec)                   # (1, n_mels, n_frames)
    log_mel = T.AmplitudeToDB()(mel)              # (1, n_mels, n_frames)
    return log_mel.squeeze(0)                     # (n_mels, n_frames)

In [ ]:
# Test
log_mel = power_to_log_mel(cleaned)
print(f"Input shape: {cleaned.shape}, Log-mel shape: {log_mel.shape}")
print(f"Expected: ({N_MELS}, n_frames)")

In [ ]:
# Visualize log-mel spectrogram
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
ax.imshow(torch.log10(cleaned + 1e-10).numpy(), aspect='auto', origin='lower')
ax.set_xlabel("Time Frames")
ax.set_ylabel("Frequency Bin (Hz)")
ax.set_title("Power Spectrogram")

ax = axes[1]
ax.imshow(log_mel.numpy(), aspect='auto', origin='lower')
ax.set_xlabel("Time Frames")
ax.set_ylabel("Frequency Bin (Hz)")
ax.set_title("Log-Mel Spectrogram")

plt.suptitle("Power -> Log-Mel Conversion")
plt.tight_layout()
plt.show()

### Feature Extraction Pipeline

This function would unified entry point that would combines preprocessing and feature extraction into a single call. It would accepts a raw waveform and an optional filter method, `"none"` for no filtering, `"spectral_gating"` for noise supression, or `"bandpass"` for the ablation baseline. This would returns the log-Mel spectrogram ready for the model input.

In [ ]:
def get_log_mel_spectrogram(waveform: torch.Tensor,
                            filter_method: str = "none",
                            **kwargs
                            ) -> torch.Tensor
    """Get log-mel spectrogram from waveform with optional filtering.

    Args:
        waveform (torch.Tensor): Input audio waveform of shape (n_samples,).
        filter_method (str, optional): Filtering method to apply. Options: "none", "bandpass", "spectral_gating". Defaults to "none".
        **kwargs: Additional arguments for filtering methods.

    Returns:
        torch.Tensor: Log-mel spectrogram of shape (n_mels, n_frames).
    """
    if filter_method == "none":
        power = compute_power_spectrogram(waveform)
    elif filter_method == "spectral_gating":
        power = apply_spectral_gating(
            waveform,
            omega=kwargs.get("omega", 1.5),
            delta=kwargs.get("delta", 0.02),
            noise_frame=kwargs.get("noise_frame", 5)
        )
    elif filter_method == "bandpass":
        filtered = apply_bandpass(
            waveform,
            low_freq=kwargs.get("low_freq", 300.0),
            high_freq=kwargs.get("high_freq", 3400.0),
            sample_rate=kwargs.get("sample_rate", TARGET)
        )
        power = compute_power_spectrogram(filtered)
    else:
        raise ValueError(f"Invalid filter_method: {filter_method}. Choose from 'none', 'bandpass', 'spectral_gating'.")
    
    log_mel = power_to_log_mel(power)
    return log_mel.unsqueeze(0)  # (n_mels, n_frames) -> (1, n_mels, n_frames)

In [ ]:
# Test all methods
print("Output shapes for different filtering methods:")
for method in ["none", "bandpass", "spectral_gating"]:
    output = get_log_mel_spectrogram(noisy, filter_method=method)
    print(f"{method}: {output.shape}")

In [ ]:
# Visualize all methods
methods = ["none", "bandpass", "spectral_gating"]
labels = ["No Filtering", "Bandpass Filter", "Spectral Gating"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, method, label in zip(axes, methods, labels):
    log_mel = get_log_mel_spectrogram(noisy, filter_method=method).squeeze(0)
    ax.imshow(log_mel.numpy(), aspect='auto', origin='lower')
    ax.set_title(label)
    ax.set_xlabel("Time Frames")
    ax.set_ylabel("Mel Frequency Bin")

plt.suptitle("Log-Mel Spectrogram Comparison")
plt.tight_layout()
plt.show()